In [0]:
print("Spark Version:", spark.version)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Define the schema [cite: 322-327]
schema = StructType([
    StructField("flight_id", StringType(), True),
    StructField("aircraft_type", StringType(), True),
    StructField("delay_minutes", IntegerType(), True),
    StructField("status", StringType(), True)
])

# Dummy data with a duplicate and a NULL [cite: 544, 404]
data = [
    ("FL001", "Boeing 737", 75, "COMPLETED"),
    ("FL002", "Airbus A320", 15, "COMPLETED"),
    ("FL003", "Boeing 787", None, "CANCELLED"),
    ("FL001", "Boeing 737", 75, "COMPLETED"), # Duplicate [cite: 544]
    ("FL004", "Airbus A350", 45, "COMPLETED")
]

# Create DataFrame
df = spark.createDataFrame(data, schema)
df.show()

In [0]:
# 1. Drop duplicates [cite: 544]
# 2. Fill NULL delays with 0 [cite: 413, 547-548]
silver_df = df.dropDuplicates().fillna(0, subset=["delay_minutes"])

silver_df.show()

In [0]:
# Create a Risk Tier based on delay minutes [cite: 433-436, 549-552]
triage_df = silver_df.withColumn("risk_tier", 
    F.when(F.col("delay_minutes") > 60, "HIGH")
    .when(F.col("delay_minutes") > 20, "MEDIUM")
    .otherwise("LOW")
)

triage_df.show()

In [0]:
# Group by aircraft type and find average delay [cite: 558-561, 356]
gold_df = triage_df.groupBy("aircraft_type").agg(
    F.count("flight_id").alias("total_flights"), # [cite: 354, 559]
    F.avg("delay_minutes").alias("avg_delay")   # [cite: 356, 561]
)

gold_df.show()

In [0]:
from pyspark.sql import Window

# Define the window [cite: 369-370]
windowSpec = Window.partitionBy("aircraft_type").orderBy(F.desc("delay_minutes"))

# Apply Rank [cite: 374]
ranked_df = triage_df.withColumn("rank", F.rank().over(windowSpec))

ranked_df.filter(F.col("rank") == 1).show()